In [1]:
from ast import literal_eval
import pandas as pd
from pathlib import Path
import torch
import torch.nn as nn
import pytorch_lightning as pl
from torch.utils.data import TensorDataset
import configparser
import time

from Losses import shannon_entropy_loss
import sys

# init device
if torch.cuda.is_available():
    device = torch.device("cuda:0")
    print(f"Using {torch.cuda.get_device_name()} for training.")
else:
    device = torch.device("cpu")
    print("No GPU available, using CPU for training.")
    
torch.set_float32_matmul_precision('high')  # For performance



config = configparser.ConfigParser()
config.read("model_setting.ini")

raw_path = '/media/warmachine/DBDISK/SSFocus/data/Processed/Sao_Paolo/raw_s1b-s6-raw-s-vv-20210103t214313-20210103t214344-024995-02f995.pkl'
focus_path = '/media/warmachine/DBDISK/SSFocus/data/Processed/Sao_Paolo/focused_raw_s1b-s6-raw-s-vv-20210103t214313-20210103t214344-024995-02f995.pkl'

raw = pd.read_pickle(raw_path)
focused = pd.read_pickle(focus_path)

aux = raw['metadata']

    

Using NVIDIA A40-24C for training.


FileNotFoundError: [Errno 2] No such file or directory: '/media/warmachine/DBDISK/SSFocus/data/Processed/Sao_Paolo/raw_s1b-s6-raw-s-vv-20210103t214313-20210103t214344-024995-02f995.pkl'

In [2]:
meta = pd.read_pickle('Data/RAW/SM/numpy/s1a-s1-raw-s-vv-20200509t183227-20200509t183238-032492-03c34a_pkt_8_metadata.pkl')

In [3]:
import sentinel1decoder


def load_constants_from_meta(meta, patch_dim = [2000,4000]):
    constants = {}
    constants['wavelength'] = sentinel1decoder.constants.TX_WAVELENGTH_M
    constants['c'] = sentinel1decoder.constants.SPEED_OF_LIGHT_MPS  # Speed of light
    constants['PRI'] = meta["PRI"].unique()[0]
    constants['len_az_line'] = patch_dim[0]
    constants['len_range_line'] = patch_dim[1]
    constants['az_sample_freq'] = 1 / constants['PRI']
    RGDEC = meta["Range Decimation"].unique()[0]
    constants['range_sample_freq'] = sentinel1decoder.utilities.range_dec_to_sample_rate(RGDEC)
    constants['range_sample_period'] = 1 / constants['range_sample_freq']
    constants['pi'] = 3.141592653589793
    constants['rank'] = meta["Rank"].unique()[0]
    suppressed_data_time = 320/(8*sentinel1decoder.constants.F_REF)

    
    constants['range_start_time'] = meta["SWST"].unique()[0] + suppressed_data_time
    
    
    # TODO: verify the correct application of RST
    # print('Before:')
    # print(constants['range_start_time'])
    # print('After:')
    # constants['range_start_time'] = meta["SWST"].unique()[0] + suppressed_data_time + X_RANGE[1] * constants['range_sample_period']
    # print(constants['range_start_time'])
    scale_factor = constants['len_range_line']/19950
    
    
    
    constants['sample_num_along_range_line'] = torch.arange(start=0, end=constants['len_range_line'], step=1, dtype=torch.float64)
    fast_time_vec = constants['range_start_time'] + (constants['range_sample_period'] * constants['sample_num_along_range_line'])
    constants['fast_time_vec'] = fast_time_vec
    
    
    
    # REPLICA:
    TXPSF = meta["Tx Pulse Start Frequency"].unique()[0]
    TXPRR = meta["Tx Ramp Rate"].unique()[0] 
    print("TXPSF:",TXPSF)
    print("TXPRR:",TXPRR)

    TXPL = meta["Tx Pulse Length"].unique()[0] 
    print("Pulse Lenght:",TXPL)
    num_tx_vals = int(TXPL*constants['range_sample_freq'])
    tx_replica_time_vals = torch.linspace(-TXPL/2, TXPL/2, steps=num_tx_vals)
    phi1 = TXPSF + TXPRR*TXPL/2
    phi2 = TXPRR/2
    tx_replica = torch.exp(2j * constants['pi'] * (phi1*tx_replica_time_vals + phi2*tx_replica_time_vals**2))
    constants['tx_replica'] = tx_replica
    return constants


meta2 = load_constants_from_meta(meta)

TXPSF: -43801344.99740284
TXPRR: 1927378686406.996
Pulse Lenght: 4.5451248822135946e-05


In [14]:
for key in meta2.keys():
    print(key, meta2[key])
    
    
    
    


wavelength 0.055465764662349676
c 299792458.0
PRI 0.0005345716926237736
len_az_line 2000
len_range_line 4000
az_sample_freq 1870.6564784450536
range_sample_freq 100092592.64
range_sample_period 9.990749301465992e-09
pi 3.141592653589793
rank 9
range_start_time 0.00011450730799386887
sample_num_along_range_line tensor([0.0000e+00, 1.0000e+00, 2.0000e+00,  ..., 3.9970e+03, 3.9980e+03,
        3.9990e+03], dtype=torch.float64)
fast_time_vec tensor([0.0001, 0.0001, 0.0001,  ..., 0.0002, 0.0002, 0.0002],
       dtype=torch.float64)
tx_replica tensor([-0.2383-0.9712j, -0.1504+0.9886j,  0.5176-0.8556j,  ...,
         0.4008-0.9162j, -0.0194+0.9998j, -0.3637-0.9315j])
